# Publication notebook: final SPAdes comparison summaries

- Purpose: summarize final assembly-comparison outputs from the publication workflow.
- Required inputs: local `data/assemblies_spades/final_run/` directories and `results/tables/eval_final_all_t0p5.tsv`.
- Required models: none at notebook runtime if the retained TSV already exists.
- Required external tools: prior `spades.py` outputs must already exist locally.
- Expected outputs: final assembly summary TSVs under `results/tables/`.
- Publication output: final assembly comparison tables.
- Reproduction status: normalized for portability, but remains local-only because assembly outputs are not tracked.


In [ ]:
from pathlib import Path
import re
import csv
import gzip
import ast
import pandas as pd
from functools import lru_cache

PROJECT_ROOT = Path("..").resolve()
ASSEMBLIES_ROOT = PROJECT_ROOT / "data" / "assemblies_spades" / "final_run"
OUT_TSV = PROJECT_ROOT / "results" / "tables" / "eval_final_all_t0p5.tsv"

print("PROJECT_ROOT   =", PROJECT_ROOT)
print("ASSEMBLIES_ROOT=", ASSEMBLIES_ROOT)
print("OUT_TSV        =", OUT_TSV)

In [ ]:
def read_text(p: Path) -> str:
    try:
        return p.read_text(errors="ignore")
    except Exception:
        return ""

def fasta_contig_lengths(fa_path: Path):
    lens = []
    cur = 0
    for line in fa_path.read_text(errors="ignore").splitlines():
        if line.startswith(">"):
            if cur > 0:
                lens.append(cur)
                cur = 0
        else:
            cur += len(line.strip())
    if cur > 0:
        lens.append(cur)
    return lens

@lru_cache(maxsize=None)
def count_fastq_reads(fq_path_str: str):
    fq_path = Path(fq_path_str)
    if not fq_path.exists():
        return None
    opener = gzip.open if fq_path.suffix == ".gz" else open
    n_lines = 0
    with opener(fq_path, "rt", errors="ignore") as f:
        for _ in f:
            n_lines += 1
    return n_lines // 4 if n_lines else 0

def parse_spades_fastqs(spades_dir: Path):
    candidates = [
        spades_dir / "spades.log",
        spades_dir / "spades.log.txt",
        spades_dir / "params.txt",
    ]
    txt = ""
    for c in candidates:
        if c.exists():
            txt += "\n" + read_text(c)

    left_path = None
    right_path = None

    m_left = re.search(r"left reads:\s*(\[[^\n]+\])", txt, flags=re.I)
    m_right = re.search(r"right reads:\s*(\[[^\n]+\])", txt, flags=re.I)

    if m_left:
        try:
            vals = ast.literal_eval(m_left.group(1))
            if isinstance(vals, list) and vals:
                left_path = vals[0]
        except Exception:
            pass

    if m_right:
        try:
            vals = ast.literal_eval(m_right.group(1))
            if isinstance(vals, list) and vals:
                right_path = vals[0]
        except Exception:
            pass

    reads_pairs_used = count_fastq_reads(left_path) if left_path else None

    return {
        "left_reads_path": left_path,
        "right_reads_path": right_path,
        "reads_pairs_used": reads_pairs_used,
    }

In [ ]:
def parse_folder_name(folder: str):
    """
    Returns:
      dataset, model, thresh

    Supports old GB/CNN naming and new BIGRU naming.
    """

    # old scheme: 10K_final_10_unfiltered / gb / cnn_fixedep25
    m = re.match(r"^(3200|10K|20K)_final_(\d+)_(.+)$", folder, flags=re.I)
    if m:
        read_level = m.group(1).upper()
        chim = m.group(2)
        suffix = m.group(3).lower()

        dataset = f"{read_level}_final_{chim}"

        if suffix == "unfiltered":
            return dataset, "UNFILTERED", "NA"
        if suffix.startswith("gb"):
            m_t = re.search(r"t([0-9]+p?[0-9]*)", suffix)
            thresh = m_t.group(1).replace("p", ".") if m_t else "0.5"
            return dataset, "GB", thresh
        if suffix.startswith("cnn"):
            m_t = re.search(r"t([0-9]+p?[0-9]*)", suffix)
            thresh = m_t.group(1).replace("p", ".") if m_t else "0.5"
            return dataset, "CNN_fixedep25", thresh

        return None, None, None

    # new bigru scheme: bigru_10k_10_bigru_filtered / bigru_10k_10_unfiltered
    m = re.match(r"^bigru_(10k|20k|3k)_(\d+)_(bigru_filtered|unfiltered)$", folder, flags=re.I)
    if m:
        rl_raw = m.group(1).lower()
        chim = m.group(2)
        suffix = m.group(3).lower()

        rl_map = {"3k": "3200", "10k": "10K", "20k": "20K"}
        read_level = rl_map[rl_raw]
        dataset = f"{read_level}_final_{chim}"

        if suffix == "unfiltered":
            return dataset, "UNFILTERED", "NA"
        if suffix == "bigru_filtered":
            return dataset, "BIGRU", "0.5"

    return None, None, None

In [ ]:
rows = []

for run_root in sorted([p for p in ASSEMBLIES_ROOT.iterdir() if p.is_dir()]):
    dataset, model, thresh = parse_folder_name(run_root.name)
    if dataset is None:
        continue

    contig_hits = list(run_root.rglob("contigs.fasta"))
    if not contig_hits:
        rows.append({
            "dataset": dataset,
            "model": model,
            "thresh": thresh,
            "reads_out": None,
            "spades_contigs": None,
            "spades_total_len": None,
            "spades_max_len": None,
            "contigs_path": None,
            "run_root": str(run_root),
        })
        continue

    contigs = contig_hits[0]
    lens = fasta_contig_lengths(contigs)
    spades_contigs = len(lens)
    spades_total_len = sum(lens) if lens else 0
    spades_max_len = max(lens) if lens else None

    spades_dir = contigs.parent
    log_info = parse_spades_fastqs(spades_dir)
    reads_out = log_info["reads_pairs_used"]

    rows.append({
        "dataset": dataset,
        "model": model,
        "thresh": thresh,
        "reads_out": reads_out,
        "spades_contigs": spades_contigs,
        "spades_total_len": spades_total_len,
        "spades_max_len": spades_max_len,
        "contigs_path": str(contigs.relative_to(PROJECT_ROOT)),
        "run_root": str(run_root),
    })

raw_df = pd.DataFrame(rows).sort_values(["dataset", "model"]).reset_index(drop=True)
raw_df.head(20)

In [ ]:
raw_df[["dataset", "model"]].value_counts().sort_index()

In [ ]:
df = (
    raw_df.sort_values(
        ["dataset", "model", "reads_out", "spades_total_len"],
        ascending=[True, True, False, False]
    )
    .drop_duplicates(subset=["dataset", "model"], keep="first")
    .reset_index(drop=True)
)

df[["dataset", "model", "reads_out", "spades_total_len", "contigs_path"]].sort_values(["dataset", "model"])

In [ ]:
base_reads = (
    df[df["model"] == "UNFILTERED"][["dataset", "reads_out"]]
    .rename(columns={"reads_out": "reads_in"})
)

eval_df = df.merge(base_reads, on="dataset", how="left")
eval_df["reads_out"] = pd.to_numeric(eval_df["reads_out"], errors="coerce")
eval_df["reads_in"] = pd.to_numeric(eval_df["reads_in"], errors="coerce")

eval_df["pct_kept"] = ((eval_df["reads_out"] / eval_df["reads_in"]) * 100).round(3)
eval_df.loc[eval_df["model"] == "UNFILTERED", "pct_kept"] = 100.000

eval_df[["dataset", "model", "reads_in", "reads_out", "pct_kept"]].sort_values(["dataset", "model"])

In [ ]:
final_eval = eval_df[[
    "dataset",
    "model",
    "thresh",
    "reads_in",
    "reads_out",
    "pct_kept",
    "spades_contigs",
    "spades_total_len",
    "spades_max_len",
    "contigs_path",
]].sort_values(["dataset", "model"]).reset_index(drop=True)

final_eval

In [ ]:
print(final_eval["model"].value_counts(dropna=False))
print()
display(final_eval.head(20))

In [ ]:
OUT_TSV.parent.mkdir(parents=True, exist_ok=True)
final_eval.to_csv(OUT_TSV, sep="\t", index=False)
print("Wrote:", OUT_TSV)

In [ ]:
ALT_TSV = PROJECT_ROOT / "results" / "tables" / "eval_final_all_t0p5.tsv"
final_eval.to_csv(ALT_TSV, sep="\t", index=False)
print("Wrote:", ALT_TSV)

# FINAL

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

# --- find repo root by walking up until we see both "data/" and "results/" ---
cwd = Path.cwd()
root = None
for p in [cwd] + list(cwd.parents):
    if (p / "data").exists() and (p / "results").exists():
        root = p
        break

# fallback: only require "data/"
if root is None:
    for p in [cwd] + list(cwd.parents):
        if (p / "data").exists():
            root = p
            break

if root is None:
    raise FileNotFoundError(f"Could not locate repo root from cwd={cwd}")

print("Repo root:", root)

preferred = root / "results" / "tables" / "eval_final_all_t0p5.tsv"
if preferred.exists():
    tsv = preferred
else:
    hits = list(root.rglob("eval_final_all_t0p5.tsv"))
    if not hits:
        raise FileNotFoundError(f"Could not find eval_final_all_t0p5.tsv under {root}")
    tsv = hits[0]

print("Loading TSV:", tsv)

df = pd.read_csv(tsv, sep="\t")
df.head()

In [ ]:
df["model"] = df["model"].astype(str).str.upper().str.strip()

df["model"] = df["model"].replace({
    "CNN_FIXEDEP25": "CNN",
    "BIGRU": "BIGRU",
    "UNFILTERED": "UNFILTERED",
    "GB": "GB",
})

for c in ["reads_in", "reads_out", "pct_kept", "spades_contigs", "spades_total_len", "spades_max_len"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

print(df["model"].value_counts(dropna=False))
df[["dataset", "model"]].drop_duplicates().sort_values(["dataset", "model"]).head(20)

In [ ]:
def parse_read_level(s: str):
    m = re.match(r"^(3200|10K|20K)", str(s))
    return m.group(1) if m else None

def parse_chimera_pct(s: str):
    m = re.search(r"_final_(\d+)", str(s))
    return int(m.group(1)) if m else None

df["read_level"] = df["dataset"].apply(parse_read_level)
df["chimera_pct"] = df["dataset"].apply(parse_chimera_pct)

order = ["3200", "10K", "20K"]
df["read_level"] = pd.Categorical(df["read_level"], order, ordered=True)

df[["dataset", "read_level", "chimera_pct", "model", "thresh", "pct_kept"]].sort_values(
    ["read_level", "chimera_pct", "model"]
).head(20)

In [ ]:
def contigs_to_gfa(contigs_path: str) -> Path | None:
    p = Path(contigs_path)
    contigs_abs = (root / p) if not p.is_absolute() else p
    spades_dir = contigs_abs.parent

    candidates = [
        spades_dir / "assembly_graph_with_scaffolds.gfa",
        spades_dir / "assembly_graph.gfa",
        spades_dir / "assembly_graph_with_scaffolds.gfa.gz",
        spades_dir / "assembly_graph.gfa.gz",
    ]

    for gp in candidates:
        if gp.exists():
            return gp
    return None


def gfa_metrics(gfa_path: Path):
    """
    nodes = #segments (S lines)
    links = #links (L lines)
    components = connected components (undirected)
    """
    segs = set()
    edges = []
    nS = nL = 0

    if gfa_path.suffix == ".gz":
        import gzip
        opener = gzip.open
    else:
        opener = open

    with opener(gfa_path, "rt") as f:
        for line in f:
            if not line or line[0] == "#":
                continue
            if line.startswith("S\t"):
                parts = line.rstrip("\n").split("\t")
                segs.add(parts[1])
                nS += 1
            elif line.startswith("L\t"):
                parts = line.rstrip("\n").split("\t")
                u, v = parts[1], parts[3]
                edges.append((u, v))
                nL += 1

    parent = {x: x for x in segs}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for u, v in edges:
        if u in parent and v in parent:
            union(u, v)

    comps = {find(x) for x in segs} if segs else set()
    return nS, nL, len(comps)

In [ ]:
gfa_nodes, gfa_links, gfa_components, gfa_paths = [], [], [], []

for cp in df["contigs_path"].astype(str):
    gp = contigs_to_gfa(cp)
    gfa_paths.append(str(gp) if gp is not None else None)

    if gp is not None and gp.exists():
        nS, nL, nC = gfa_metrics(gp)
    else:
        nS, nL, nC = (np.nan, np.nan, np.nan)

    gfa_nodes.append(nS)
    gfa_links.append(nL)
    gfa_components.append(nC)

df["gfa_path"] = gfa_paths
df["gfa_nodes"] = gfa_nodes
df["gfa_links"] = gfa_links
df["gfa_components"] = gfa_components

df[["dataset", "model", "gfa_path", "gfa_nodes", "gfa_components"]].head(10)

In [ ]:
final_cols = [
    "read_level", "chimera_pct", "dataset", "model", "thresh",
    "reads_in", "reads_out", "pct_kept",
    "spades_contigs", "spades_total_len", "spades_max_len",
    "gfa_components", "gfa_nodes", "gfa_links",
    "contigs_path", "gfa_path"
]

final_tbl = df[final_cols].sort_values(["read_level", "chimera_pct", "model"])
final_tbl

In [ ]:
base = (
    df[df["model"] == "UNFILTERED"]
    .set_index("dataset")[[
        "spades_contigs", "spades_total_len", "spades_max_len",
        "gfa_components", "gfa_nodes", "gfa_links"
    ]]
    .rename(columns=lambda c: "unf_" + c)
)

dfd = df.join(base, on="dataset")

dfd["delta_contigs"] = dfd["spades_contigs"] - dfd["unf_spades_contigs"]
dfd["delta_total_len"] = dfd["spades_total_len"] - dfd["unf_spades_total_len"]
dfd["delta_max_len"] = dfd["spades_max_len"] - dfd["unf_spades_max_len"]
dfd["delta_components"] = dfd["gfa_components"] - dfd["unf_gfa_components"]
dfd["delta_nodes"] = dfd["gfa_nodes"] - dfd["unf_gfa_nodes"]

In [ ]:
delta_tbl = dfd[dfd["model"].isin(["GB", "CNN", "BIGRU"])][[
    "read_level", "chimera_pct", "dataset", "model", "thresh", "pct_kept",
    "delta_components", "delta_nodes", "delta_contigs", "delta_max_len", "delta_total_len"
]].sort_values(["read_level", "chimera_pct", "model"])

delta_tbl

In [ ]:
cand = dfd[dfd["model"].isin(["GB", "CNN", "BIGRU"])].copy()

best = (
    cand.sort_values(
        ["dataset", "gfa_components", "gfa_nodes", "spades_contigs", "spades_max_len", "pct_kept"],
        ascending=[True, True, True, True, False, False]
    )
    .groupby("dataset", as_index=False)
    .head(1)
    .sort_values(["read_level", "chimera_pct"])
)

best[[
    "read_level", "chimera_pct", "dataset", "model", "thresh", "pct_kept",
    "spades_contigs", "spades_max_len", "gfa_components", "gfa_nodes",
    "delta_components", "delta_nodes", "delta_contigs", "delta_max_len"
]]

In [ ]:
plot_df = dfd[dfd["model"].isin(["UNFILTERED", "GB", "CNN", "BIGRU"])].copy()
model_order = ["UNFILTERED", "GB", "CNN", "BIGRU"]

def plot_metric(metric, ylabel, title_suffix):
    for rl in ["3200", "10K", "20K"]:
        d = plot_df[plot_df["read_level"] == rl].sort_values("chimera_pct")
        if d.empty:
            continue

        plt.figure(figsize=(7, 4))
        for m in model_order:
            dm = d[d["model"] == m]
            if dm.empty:
                continue
            plt.plot(dm["chimera_pct"], dm[metric], marker="o", label=m)

        plt.xlabel("Chimeric %")
        plt.ylabel(ylabel)
        plt.title(f"{rl}: {title_suffix}")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.show()

plot_metric("gfa_components", "GFA components", "components vs chimera%")
plot_metric("gfa_nodes", "GFA nodes", "nodes vs chimera%")
plot_metric("spades_max_len", "Max contig length", "max contig length vs chimera%")

In [ ]:
out_dir = root / "reports"
final_tbl.to_csv(out_dir / "final_metrics_with_nodes.tsv", sep="\t", index=False)
delta_tbl.to_csv(out_dir / "final_deltas_vs_unfiltered.tsv", sep="\t", index=False)
best.to_csv(out_dir / "final_best_per_dataset.tsv", sep="\t", index=False)

print("Wrote:")
print("-", out_dir / "final_metrics_with_nodes.tsv")
print("-", out_dir / "final_deltas_vs_unfiltered.tsv")
print("-", out_dir / "final_best_per_dataset.tsv")